## Imports and configuration

In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys

In [33]:
# 60 min

# "BU_TotActPwr_SDB_EL_Substation": Path(r"C:\Data_analysis\Thesis\outputs\profile_forecasting\predictions\run_20260527_083211_c3cb7819_ridge_BU_TotActPwr_SDB_EL_Substation_profile_predictions.csv"),
#         "BU_TotActPwr_Academy": Path(r"C:\Data_analysis\Thesis\outputs\profile_forecasting\predictions\run_20260521_093036_4a85546b_xgboost_BU_TotActPwr_Academy_profile_predictions.csv"),
#         "BU_TotActPwr_Tech_Room": Path(r"C:\Data_analysis\Thesis\outputs\profile_forecasting\predictions\run_20260521_111054_750e3408_xgboost_BU_TotActPwr_Tech_Room_profile_predictions.csv"),



    "30min": {
        "BU_TotActPwr_SDB_EL_Substation": Path(r"C:\Data_analysis\Thesis\outputs\profile_forecasting\predictions\run_20260527_092954_174e032d_random_forest_BU_TotActPwr_SDB_EL_Substation_profile_predictions.csv"),
        "BU_TotActPwr_Academy": Path(r"C:\Data_analysis\Thesis\outputs\profile_forecasting\predictions\run_20260521_144150_088cccee_ridge_BU_TotActPwr_Academy_profile_predictions.csv"),
        "BU_TotActPwr_Tech_Room": Path(r"C:\Data_analysis\Thesis\outputs\profile_forecasting\predictions\run_20260521_111054_750e3408_xgboost_BU_TotActPwr_Tech_Room_profile_predictions.csv"),

In [37]:
# Forecast settings
STEPS = 48
STEP_MINUTES = 30

# Prediction folder
PRED_DIR = Path(r"C:\Data_analysis\Thesis\outputs\profile_forecasting\predictions")

# Forecast prediction files
prediction_files = {
    "Academy": PRED_DIR / "run_20260521_144150_088cccee_ridge_BU_TotActPwr_Academy_profile_predictions.csv",
    "Tech_Room": PRED_DIR / "run_20260521_111054_750e3408_xgboost_BU_TotActPwr_Tech_Room_profile_predictions.csv",
    "SDB_EL_Substation": PRED_DIR / "run_20260527_092954_174e032d_random_forest_BU_TotActPwr_SDB_EL_Substation_profile_predictions.csv",
}

## Convert one prediction file from wide to long format

In [38]:
def profile_prediction_to_long(path, target_name, steps=96, step_minutes=15):

    df = pd.read_csv(path)

    df["Time"] = pd.to_datetime(df["Time"], errors="coerce")
    df = df.dropna(subset=["Time"]).sort_values("Time")

    rows = []

    for _, row in df.iterrows():
        issue_time = row["Time"]

        for step in range(1, steps + 1):
            true_col = f"y_tplus_{step:03d}_true"
            pred_col = f"y_tplus_{step:03d}_pred"

            rows.append({
                "issue_time": issue_time,
                "horizon_step": step,
                "Time": issue_time + pd.Timedelta(minutes=step_minutes * step),
                f"{target_name}_true": row[true_col],
                f"{target_name}_pred": row[pred_col],
            })

    return pd.DataFrame(rows)

## Combine all target forecast files

In [39]:
combined = None

for target_name, path in prediction_files.items():
    df_long = profile_prediction_to_long(
        path=path,
        target_name=target_name,
        steps=STEPS,
        step_minutes=STEP_MINUTES,
    )

    if combined is None:
        combined = df_long
    else:
        combined = combined.merge(
            df_long,
            on=["issue_time", "horizon_step", "Time"],
            how="outer"
        )

combined = combined.sort_values(["issue_time", "horizon_step"]).reset_index(drop=True)

combined.head()

KeyError: 'y_tplus_025_true'

## Remove incomplete forecast days

In [40]:
combined["issue_time"] = pd.to_datetime(combined["issue_time"], errors="coerce")
combined["Time"] = pd.to_datetime(combined["Time"], errors="coerce")

combined["issue_date"] = combined["issue_time"].dt.date

required_cols = [
    c for c in combined.columns
    if c.endswith("_true") or c.endswith("_pred")
]

bad_issue_dates = (
    combined
    .groupby("issue_date")[required_cols]
    .apply(lambda x: x.isna().any().any())
)

bad_issue_dates = bad_issue_dates[bad_issue_dates].index.tolist()

print("Incomplete issue dates removed:")
print(bad_issue_dates)

combined_clean = combined[~combined["issue_date"].isin(bad_issue_dates)].copy()
combined_clean = combined_clean.drop(columns=["issue_date"])

combined_clean = combined_clean.sort_values(["issue_time", "horizon_step"]).reset_index(drop=True)

print("Before cleaning:", combined.shape)
print("After cleaning:", combined_clean.shape)
print("Dropped rows:", combined.shape[0] - combined_clean.shape[0])

combined_clean.head()

Incomplete issue dates removed:
[]
Before cleaning: (912, 6)
After cleaning: (912, 5)
Dropped rows: 0


,issue_time,horizon_step,Time,Academy_true,Academy_pred
0,2026-04-11 23:30:00,1,2026-04-12 00:00:00,3.70433,4.187934
1,2026-04-11 23:30:00,2,2026-04-12 00:30:00,3.42017,3.563671
2,2026-04-11 23:30:00,3,2026-04-12 01:00:00,3.45750,3.082005
3,2026-04-11 23:30:00,4,2026-04-12 01:30:00,3.29383,2.648917
4,2026-04-11 23:30:00,5,2026-04-12 02:00:00,3.36667,2.815014


## Create total actual and forecast load

In [41]:
target_pred_cols = [
    c for c in combined_clean.columns
    if c.endswith("_pred") and c != "total_load_pred"
]

target_true_cols = [
    c for c in combined_clean.columns
    if c.endswith("_true") and c != "total_load_true"
]

combined_clean["total_load_pred"] = combined_clean[target_pred_cols].sum(axis=1)
combined_clean["total_load_true"] = combined_clean[target_true_cols].sum(axis=1)

combined_clean.head()

,issue_time,horizon_step,Time,Academy_true,Academy_pred,total_load_pred,total_load_true
0,2026-04-11 23:30:00,1,2026-04-12 00:00:00,3.70433,4.187934,4.187934,3.70433
1,2026-04-11 23:30:00,2,2026-04-12 00:30:00,3.42017,3.563671,3.563671,3.42017
2,2026-04-11 23:30:00,3,2026-04-12 01:00:00,3.45750,3.082005,3.082005,3.45750
3,2026-04-11 23:30:00,4,2026-04-12 01:30:00,3.29383,2.648917,2.648917,3.29383
4,2026-04-11 23:30:00,5,2026-04-12 02:00:00,3.36667,2.815014,2.815014,3.36667


## Create current EMS constant-hold baseline

In [42]:
df_ems = combined_clean.copy()

df_ems["issue_time"] = pd.to_datetime(df_ems["issue_time"], errors="coerce")
df_ems["Time"] = pd.to_datetime(df_ems["Time"], errors="coerce")

df_ems = df_ems.sort_values(["issue_time", "horizon_step"]).reset_index(drop=True)

In [43]:
# Current EMS baseline:
# EMS assumes the latest available load remains constant over the full 24-hour horizon.

constant_hold_per_issue = (
    df_ems
    .groupby("issue_time")["total_load_true"]
    .first()
)

df_ems["current_ems_constant_hold_load"] = df_ems["issue_time"].map(constant_hold_per_issue)

df_ems.head()

,issue_time,horizon_step,Time,Academy_true,Academy_pred,total_load_pred,total_load_true,current_ems_constant_hold_load
0,2026-04-11 23:30:00,1,2026-04-12 00:00:00,3.70433,4.187934,4.187934,3.70433,3.70433
1,2026-04-11 23:30:00,2,2026-04-12 00:30:00,3.42017,3.563671,3.563671,3.42017,3.70433
2,2026-04-11 23:30:00,3,2026-04-12 01:00:00,3.45750,3.082005,3.082005,3.45750,3.70433
3,2026-04-11 23:30:00,4,2026-04-12 01:30:00,3.29383,2.648917,2.648917,3.29383,3.70433
4,2026-04-11 23:30:00,5,2026-04-12 02:00:00,3.36667,2.815014,2.815014,3.36667,3.70433


## Calculate residuals and errors

In [44]:
# Forecast residual
df_ems["forecast_residual"] = (
    df_ems["total_load_true"] - df_ems["total_load_pred"]
)

# Current EMS baseline residual
df_ems["ems_baseline_residual"] = (
    df_ems["total_load_true"] - df_ems["current_ems_constant_hold_load"]
)

# Absolute errors
df_ems["forecast_abs_error"] = df_ems["forecast_residual"].abs()
df_ems["ems_baseline_abs_error"] = df_ems["ems_baseline_residual"].abs()

# Squared errors
df_ems["forecast_squared_error"] = df_ems["forecast_residual"] ** 2
df_ems["ems_baseline_squared_error"] = df_ems["ems_baseline_residual"] ** 2

# Absolute percentage errors
df_ems["forecast_ape"] = np.where(
    df_ems["total_load_true"] != 0,
    df_ems["forecast_abs_error"] / df_ems["total_load_true"].abs() * 100,
    np.nan
)

df_ems["ems_baseline_ape"] = np.where(
    df_ems["total_load_true"] != 0,
    df_ems["ems_baseline_abs_error"] / df_ems["total_load_true"].abs() * 100,
    np.nan
)

df_ems.head()

,issue_time,horizon_step,Time,Academy_true,Academy_pred,total_load_pred,total_load_true,current_ems_constant_hold_load,forecast_residual,ems_baseline_residual,forecast_abs_error,ems_baseline_abs_error,forecast_squared_error,ems_baseline_squared_error,forecast_ape,ems_baseline_ape
0,2026-04-11 23:30:00,1,2026-04-12 00:00:00,3.70433,4.187934,4.187934,3.70433,3.70433,-0.483604,0.00000,0.483604,0.00000,0.233873,0.000000,13.055096,0.000000
1,2026-04-11 23:30:00,2,2026-04-12 00:30:00,3.42017,3.563671,3.563671,3.42017,3.70433,-0.143501,-0.28416,0.143501,0.28416,0.020593,0.080747,4.195728,8.308359
2,2026-04-11 23:30:00,3,2026-04-12 01:00:00,3.45750,3.082005,3.082005,3.45750,3.70433,0.375495,-0.24683,0.375495,0.24683,0.140997,0.060925,10.860305,7.138973
3,2026-04-11 23:30:00,4,2026-04-12 01:30:00,3.29383,2.648917,2.648917,3.29383,3.70433,0.644913,-0.41050,0.644913,0.41050,0.415913,0.168510,19.579430,12.462695
4,2026-04-11 23:30:00,5,2026-04-12 02:00:00,3.36667,2.815014,2.815014,3.36667,3.70433,0.551656,-0.33766,0.551656,0.33766,0.304325,0.114014,16.385818,10.029495


## Global comparison: forecast vs current EMS baseline

In [45]:
global_comparison = pd.DataFrame({
    "case": [
        "Load forecasting input",
        "Current EMS constant-hold input",
    ],
    "MAE_kW": [
        df_ems["forecast_abs_error"].mean(),
        df_ems["ems_baseline_abs_error"].mean(),
    ],
    "RMSE_kW": [
        np.sqrt(df_ems["forecast_squared_error"].mean()),
        np.sqrt(df_ems["ems_baseline_squared_error"].mean()),
    ],
    "MAPE_%": [
        df_ems["forecast_ape"].mean(),
        df_ems["ems_baseline_ape"].mean(),
    ],
    "Bias_kW": [
        df_ems["forecast_residual"].mean(),
        df_ems["ems_baseline_residual"].mean(),
    ],
    "Max_abs_error_kW": [
        df_ems["forecast_abs_error"].max(),
        df_ems["ems_baseline_abs_error"].max(),
    ],
})

global_comparison

,case,MAE_kW,RMSE_kW,MAPE_%,Bias_kW,Max_abs_error_kW
0,Load forecasting input,1.924897,2.870842,26.616576,0.171982,12.669377
1,Current EMS constant-hold input,3.569538,5.741114,33.269811,3.346271,22.609500


In [ ]:
forecast_mae = global_comparison.loc[
    global_comparison["case"] == "Load forecasting input", "MAE_kW"
].iloc[0]

ems_baseline_mae = global_comparison.loc[
    global_comparison["case"] == "Current EMS constant-hold input", "MAE_kW"
].iloc[0]

mae_improvement_percent = (
    (ems_baseline_mae - forecast_mae) / ems_baseline_mae * 100
)

print(f"MAE improvement of load forecasting over current EMS baseline: {mae_improvement_percent:.2f}%")

## Day-wise forecast error and best/worst day selection

In [ ]:
def calculate_daily_error(group):
    return pd.Series({
        "n_steps": len(group),

        "forecast_MAE_kW": group["forecast_abs_error"].mean(),
        "forecast_RMSE_kW": np.sqrt(group["forecast_squared_error"].mean()),
        "forecast_MAPE_%": group["forecast_ape"].mean(),
        "forecast_Bias_kW": group["forecast_residual"].mean(),
        "forecast_Max_abs_error_kW": group["forecast_abs_error"].max(),

        "ems_baseline_MAE_kW": group["ems_baseline_abs_error"].mean(),
        "ems_baseline_RMSE_kW": np.sqrt(group["ems_baseline_squared_error"].mean()),
        "ems_baseline_MAPE_%": group["ems_baseline_ape"].mean(),
        "ems_baseline_Bias_kW": group["ems_baseline_residual"].mean(),
        "ems_baseline_Max_abs_error_kW": group["ems_baseline_abs_error"].max(),
    })

In [ ]:
daily_comparison = (
    df_ems
    .groupby("issue_time")
    .apply(calculate_daily_error)
    .reset_index()
)

# Keep only complete 96-step forecast days
daily_comparison = daily_comparison[daily_comparison["n_steps"] == STEPS].copy()

daily_comparison["MAE_improvement_%"] = (
    (daily_comparison["ems_baseline_MAE_kW"] - daily_comparison["forecast_MAE_kW"])
    / daily_comparison["ems_baseline_MAE_kW"]
    * 100
)

daily_comparison = daily_comparison.sort_values("forecast_MAE_kW").reset_index(drop=True)

daily_comparison.head()

In [ ]:
best_day_row = daily_comparison.sort_values("forecast_MAE_kW", ascending=True).iloc[0]
worst_day_row = daily_comparison.sort_values("forecast_MAE_kW", ascending=False).iloc[0]

best_issue_time = best_day_row["issue_time"]
worst_issue_time = worst_day_row["issue_time"]

print("Best forecasted day based on total-load MAE:")
display(best_day_row.to_frame().T)

print("Worst forecasted day based on total-load MAE:")
display(worst_day_row.to_frame().T)

In [ ]:
best_day = (
    df_ems[df_ems["issue_time"] == best_issue_time]
    .copy()
    .sort_values("horizon_step")
)

worst_day = (
    df_ems[df_ems["issue_time"] == worst_issue_time]
    .copy()
    .sort_values("horizon_step")
)

print("Best issue time:", best_issue_time)
print("Worst issue time:", worst_issue_time)

print("Best day shape:", best_day.shape)
print("Worst day shape:", worst_day.shape)

## Plot load input comparison

In [ ]:
def plot_load_input_comparison(one_day, title_prefix):
    issue_time = one_day["issue_time"].iloc[0]

    plt.figure(figsize=(14, 5))

    plt.plot(
        one_day["Time"],
        one_day["total_load_true"],
        marker="o",
        label="Actual load"
    )

    plt.plot(
        one_day["Time"],
        one_day["total_load_pred"],
        marker="o",
        label="Load forecasting input"
    )

    plt.plot(
        one_day["Time"],
        one_day["current_ems_constant_hold_load"],
        linestyle="--",
        linewidth=2.5,
        label="Current EMS constant-hold input"
    )

    plt.title(f"{title_prefix}\nIssue Time: {issue_time}")
    plt.xlabel("Predicted Time")
    plt.ylabel("Total Load [kW]")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_load_input_comparison(
    best_day,
    "Best Forecasted Day: Actual Load vs Forecast Input vs Current EMS Baseline"
)

plot_load_input_comparison(
    worst_day,
    "Worst Forecasted Day: Actual Load vs Forecast Input vs Current EMS Baseline"
)

## Plot residual comparison

In [ ]:
def plot_residual_comparison(one_day, title_prefix):
    issue_time = one_day["issue_time"].iloc[0]

    plt.figure(figsize=(14, 5))

    plt.axhline(0, linestyle="--", linewidth=1)

    plt.plot(
        one_day["Time"],
        one_day["forecast_residual"],
        marker="o",
        label="Forecast residual"
    )

    plt.plot(
        one_day["Time"],
        one_day["ems_baseline_residual"],
        marker="o",
        label="Current EMS baseline residual"
    )

    plt.title(f"{title_prefix}\nIssue Time: {issue_time}")
    plt.xlabel("Predicted Time")
    plt.ylabel("Residual [kW] = Actual - Input")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_residual_comparison(
    best_day,
    "Residual Comparison for Best Forecasted Day"
)

plot_residual_comparison(
    worst_day,
    "Residual Comparison for Worst Forecasted Day"
)

## Plot absolute error comparison

In [ ]:
def plot_absolute_error_comparison(one_day, title_prefix):
    issue_time = one_day["issue_time"].iloc[0]

    plt.figure(figsize=(14, 5))

    plt.plot(
        one_day["Time"],
        one_day["forecast_abs_error"],
        marker="o",
        label="Forecast absolute error"
    )

    plt.plot(
        one_day["Time"],
        one_day["ems_baseline_abs_error"],
        marker="o",
        label="Current EMS baseline absolute error"
    )

    plt.title(f"{title_prefix}\nIssue Time: {issue_time}")
    plt.xlabel("Predicted Time")
    plt.ylabel("Absolute Error [kW]")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_absolute_error_comparison(
    best_day,
    "Absolute Error Comparison for Best Forecasted Day"
)

plot_absolute_error_comparison(
    worst_day,
    "Absolute Error Comparison for Worst Forecasted Day"
)

## Energy error summary

In [ ]:
def calculate_energy_error_summary(one_day, step_minutes=15):
    step_hours = step_minutes / 60

    d = one_day.copy()

    d["actual_energy_kWh"] = d["total_load_true"] * step_hours
    d["forecast_energy_kWh"] = d["total_load_pred"] * step_hours
    d["ems_baseline_energy_kWh"] = d["current_ems_constant_hold_load"] * step_hours

    d["forecast_energy_error_kWh"] = (
        d["actual_energy_kWh"] - d["forecast_energy_kWh"]
    )

    d["ems_baseline_energy_error_kWh"] = (
        d["actual_energy_kWh"] - d["ems_baseline_energy_kWh"]
    )

    actual_energy = d["actual_energy_kWh"].sum()

    summary = pd.DataFrame({
        "case": [
            "Load forecasting input",
            "Current EMS constant-hold input",
        ],
        "actual_energy_kWh": [
            actual_energy,
            actual_energy,
        ],
        "input_energy_kWh": [
            d["forecast_energy_kWh"].sum(),
            d["ems_baseline_energy_kWh"].sum(),
        ],
        "signed_energy_error_kWh": [
            d["forecast_energy_error_kWh"].sum(),
            d["ems_baseline_energy_error_kWh"].sum(),
        ],
        "absolute_energy_error_kWh": [
            d["forecast_energy_error_kWh"].abs().sum(),
            d["ems_baseline_energy_error_kWh"].abs().sum(),
        ],
    })

    summary["signed_energy_error_%"] = (
        summary["signed_energy_error_kWh"] / summary["actual_energy_kWh"] * 100
    )

    summary["absolute_energy_error_%"] = (
        summary["absolute_energy_error_kWh"] / summary["actual_energy_kWh"] * 100
    )

    return summary, d

In [ ]:
best_energy_summary, best_day_energy = calculate_energy_error_summary(
    best_day,
    step_minutes=STEP_MINUTES
)

worst_energy_summary, worst_day_energy = calculate_energy_error_summary(
    worst_day,
    step_minutes=STEP_MINUTES
)

print("Best forecasted day energy error:")
display(best_energy_summary)

print("Worst forecasted day energy error:")
display(worst_energy_summary)

## Cumulative energy error plot

In [ ]:
def plot_cumulative_energy_error(day_energy_df, title_prefix):
    d = day_energy_df.copy()

    d["forecast_cum_energy_error_kWh"] = d["forecast_energy_error_kWh"].cumsum()
    d["ems_baseline_cum_energy_error_kWh"] = d["ems_baseline_energy_error_kWh"].cumsum()

    issue_time = d["issue_time"].iloc[0]

    plt.figure(figsize=(14, 5))

    plt.axhline(0, linestyle="--", linewidth=1)

    plt.plot(
        d["Time"],
        d["forecast_cum_energy_error_kWh"],
        marker="o",
        label="Forecast cumulative energy error"
    )

    plt.plot(
        d["Time"],
        d["ems_baseline_cum_energy_error_kWh"],
        marker="o",
        label="Current EMS baseline cumulative energy error"
    )

    plt.title(f"{title_prefix}\nIssue Time: {issue_time}")
    plt.xlabel("Predicted Time")
    plt.ylabel("Cumulative Energy Error [kWh]")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_cumulative_energy_error(
    best_day_energy,
    "Cumulative Energy Error for Best Forecasted Day"
)

plot_cumulative_energy_error(
    worst_day_energy,
    "Cumulative Energy Error for Worst Forecasted Day"
)

## Final best/worst energy summary table

In [ ]:
best_energy_summary["forecast_quality_day"] = "Best forecasted day"
worst_energy_summary["forecast_quality_day"] = "Worst forecasted day"

energy_summary_best_worst = pd.concat(
    [best_energy_summary, worst_energy_summary],
    ignore_index=True
)

energy_summary_best_worst = energy_summary_best_worst[
    [
        "forecast_quality_day",
        "case",
        "actual_energy_kWh",
        "input_energy_kWh",
        "signed_energy_error_kWh",
        "absolute_energy_error_kWh",
        "signed_energy_error_%",
        "absolute_energy_error_%",
    ]
]

energy_summary_best_worst

### Bar chart: absolute energy error for best and worst day

In [ ]:
import matplotlib.pyplot as plt

plot_df = energy_summary_best_worst.copy()

# Make labels easier to read
plot_df["label"] = (
    plot_df["forecast_quality_day"] + "\n" + plot_df["case"]
)

plt.figure(figsize=(12, 5))

plt.bar(
    plot_df["label"],
    plot_df["absolute_energy_error_kWh"]
)

plt.title("Absolute Energy Error: Load Forecasting vs Current EMS Baseline")
plt.xlabel("Case")
plt.ylabel("Absolute Energy Error [kWh]")
plt.xticks(rotation=20, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### Cleaner grouped bar chart

In [ ]:
plot_df = energy_summary_best_worst.copy()

pivot_abs = plot_df.pivot(
    index="forecast_quality_day",
    columns="case",
    values="absolute_energy_error_kWh"
)

ax = pivot_abs.plot(
    kind="bar",
    figsize=(10, 5)
)

ax.set_title("Absolute Energy Error for Best and Worst Forecasted Days")
ax.set_xlabel("Forecast quality day")
ax.set_ylabel("Absolute Energy Error [kWh]")
ax.grid(axis="y", alpha=0.3)
ax.legend(title="Input case")

plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Percentage energy error bar chart

In [ ]:
pivot_pct = plot_df.pivot(
    index="forecast_quality_day",
    columns="case",
    values="absolute_energy_error_%"
)

ax = pivot_pct.plot(
    kind="bar",
    figsize=(10, 5)
)

ax.set_title("Absolute Energy Error Percentage for Best and Worst Forecasted Days")
ax.set_xlabel("Forecast quality day")
ax.set_ylabel("Absolute Energy Error [% of actual daily energy]")
ax.grid(axis="y", alpha=0.3)
ax.legend(title="Input case")

plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Signed energy error chart

In [ ]:
pivot_signed = plot_df.pivot(
    index="forecast_quality_day",
    columns="case",
    values="signed_energy_error_kWh"
)

ax = pivot_signed.plot(
    kind="bar",
    figsize=(10, 5)
)

ax.axhline(0, linestyle="--", linewidth=1)

ax.set_title("Signed Energy Error for Best and Worst Forecasted Days")
ax.set_xlabel("Forecast quality day")
ax.set_ylabel("Signed Energy Error [kWh]")
ax.grid(axis="y", alpha=0.3)
ax.legend(title="Input case")

plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
pivot_abs = energy_summary_best_worst.pivot(
    index="forecast_quality_day",
    columns="case",
    values="absolute_energy_error_kWh"
)

ax = pivot_abs.plot(
    kind="bar",
    figsize=(10, 5)
)

ax.set_title("Energy Error Comparison: Forecast Input vs Current EMS Baseline")
ax.set_xlabel("")
ax.set_ylabel("Absolute Energy Error [kWh]")
ax.grid(axis="y", alpha=0.3)
ax.legend(title="EMS input")

plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5 min vs 15 min

In [ ]:
df_5min = pd.read_csv(r"C:\Data_analysis\Thesis\Data\02_Preprocessing\LF_data_Trimed.csv")
df_5min = df_5min.set_index("Time")

df_5min = df_5min.copy()
df_5min.index = pd.to_datetime(df_5min.index, errors="coerce")
df_5min = df_5min.sort_index()

In [ ]:
df_15min = pd.read_csv(r"C:\Data_analysis\Thesis\Data\02_Preprocessing\resampled\LF_data_droped_15m.csv")
df_15min = df_15min.set_index("Time")

df_15min = df_15min.copy()
df_15min.index = pd.to_datetime(df_15min.index, errors="coerce")
df_15min = df_15min.sort_index()

In [ ]:
target_cols= [
    "BU_TotActPwr_Academy",
    "BU_TotActPwr_Tech_Room",
    "BU_TotActPwr_SDB_EL_Substation",
]

In [ ]:
target_cols_5min = target_cols
target_cols_15min = target_cols

In [ ]:
def compare_5min_15min_actual_energy(
    df_5min,
    df_15min,
    selected_issue_time,
    target_cols,
    step_minutes_5min=5,
    step_minutes_15min=15,
):
    selected_issue_time = pd.to_datetime(selected_issue_time)

    start_time = selected_issue_time + pd.Timedelta(minutes=step_minutes_15min)
    end_time = selected_issue_time + pd.Timedelta(hours=24)

    # 5-min actual data
    d5 = df_5min.copy()
    d5.index = pd.to_datetime(d5.index, errors="coerce")
    d5 = d5.sort_index()

    d5 = d5.loc[start_time:end_time].copy()
    d5 = d5[d5.index < end_time]

    d5["total_load_5min_actual"] = d5[target_cols].sum(axis=1)

    # 15-min actual data
    d15 = df_15min.copy()
    d15.index = pd.to_datetime(d15.index, errors="coerce")
    d15 = d15.sort_index()

    d15 = d15.loc[start_time:end_time].copy()
    d15 = d15[d15.index < end_time]

    d15["total_load_15min_actual"] = d15[target_cols].sum(axis=1)

    # Energy calculation
    energy_5min_kWh = d5["total_load_5min_actual"].sum() * (step_minutes_5min / 60)
    energy_15min_kWh = d15["total_load_15min_actual"].sum() * (step_minutes_15min / 60)

    diff_kWh = energy_15min_kWh - energy_5min_kWh
    diff_pct = diff_kWh / energy_5min_kWh * 100

    summary = pd.DataFrame({
        "resolution": [
            "5-min original actual",
            "15-min resampled actual",
        ],
        "samples": [
            len(d5),
            len(d15),
        ],
        "energy_kWh": [
            energy_5min_kWh,
            energy_15min_kWh,
        ],
    })

    return summary, d5, d15, diff_kWh, diff_pct

In [ ]:
def print_resolution_energy_result(day_name, issue_time, summary, diff_kWh, diff_pct):
    issue_time = pd.to_datetime(issue_time)
    start_time = issue_time + pd.Timedelta(minutes=15)
    end_time = issue_time + pd.Timedelta(hours=24)

    print("=" * 70)
    print(day_name)
    print("=" * 70)
    print(f"Issue time:        {issue_time}")
    print(f"Prediction start:  {start_time}")
    print(f"Prediction end:    {end_time}")
    print()
    display(summary)
    print()
    print(f"Energy difference, 15-min - 5-min: {diff_kWh:.3f} kWh")
    print(f"Energy difference percentage:       {diff_pct:.3f} %")
    print()

In [ ]:
target_cols = [
    "BU_TotActPwr_Academy",
    "BU_TotActPwr_Tech_Room",
    "BU_TotActPwr_SDB_EL_Substation",
]

best_resolution_summary, best_5min_day, best_15min_day, best_diff_kWh, best_diff_pct = (
    compare_5min_15min_actual_energy(
        df_5min=df_5min,
        df_15min=df_15min,
        selected_issue_time=best_issue_time,
        target_cols=target_cols,
    )
)

print_resolution_energy_result(
    day_name="Best forecasted day",
    issue_time=best_issue_time,
    summary=best_resolution_summary,
    diff_kWh=best_diff_kWh,
    diff_pct=best_diff_pct,
)

In [ ]:
worst_resolution_summary, worst_5min_day, worst_15min_day, worst_diff_kWh, worst_diff_pct = (
    compare_5min_15min_actual_energy(
        df_5min=df_5min,
        df_15min=df_15min,
        selected_issue_time=worst_issue_time,
        target_cols=target_cols,
    )
)

print_resolution_energy_result(
    day_name="Worst forecasted day",
    issue_time=worst_issue_time,
    summary=worst_resolution_summary,
    diff_kWh=worst_diff_kWh,
    diff_pct=worst_diff_pct,
)

In [ ]:
def plot_5min_vs_15min_actual_load(d5, d15, selected_issue_time):
    plt.figure(figsize=(14, 5))

    plt.plot(
        d5.index,
        d5["total_load_5min_actual"],
        label="5-min original actual total load",
        linewidth=1.2
    )

    plt.plot(
        d15.index,
        d15["total_load_15min_actual"],
        marker="o",
        label="15-min resampled actual total load",
        linewidth=1.5
    )

    plt.title(f"Actual Total Load Comparison: 5-min vs 15-min\nIssue Time: {selected_issue_time}")
    plt.xlabel("Time")
    plt.ylabel("Total Load [kW]")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_5min_vs_15min_actual_load(best_5min_day, best_15min_day, best_issue_time)
plot_5min_vs_15min_actual_load(worst_5min_day, worst_15min_day, worst_issue_time)

In [ ]:
target_cols = [
    "BU_TotActPwr_Academy",
    "BU_TotActPwr_Tech_Room",
    "BU_TotActPwr_SDB_EL_Substation",
]

## Extra understanding / diagnostic cells

### Check number of available issue times

In [ ]:
available_issues = sorted(df_ems["issue_time"].dropna().unique())

print("Number of forecast issue times:", len(available_issues))
print("First 5 issue times:")
print(available_issues[:5])

### Plot all target forecasts over the full test horizon

In [ ]:
df_plot = combined_clean.sort_values("Time").copy()

target_pred_cols = [
    c for c in df_plot.columns
    if c.endswith("_pred") and c != "total_load_pred"
]

for pred_col in target_pred_cols:
    target_name = pred_col.replace("_pred", "")
    true_col = f"{target_name}_true"

    plt.figure(figsize=(16, 5))

    plt.plot(
        df_plot["Time"],
        df_plot[true_col],
        label=f"{target_name} actual",
        linewidth=1.5
    )

    plt.plot(
        df_plot["Time"],
        df_plot[pred_col],
        label=f"{target_name} predicted",
        linewidth=1.5
    )

    plt.title(f"{target_name}: Actual vs Predicted Load Over Forecast Time")
    plt.xlabel("Forecasted Time")
    plt.ylabel("Load [kW]")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

### Best day for each individual target

In [ ]:
def calculate_day_error_for_target(group, target_name):
    true_col = f"{target_name}_true"
    pred_col = f"{target_name}_pred"

    y_true = group[true_col].values
    y_pred = group[pred_col].values

    residual = y_true - y_pred

    return pd.Series({
        "n_steps": len(group),
        "MAE": np.mean(np.abs(residual)),
        "RMSE": np.sqrt(np.mean(residual ** 2)),
        "MAPE": np.nanmean(
            np.where(
                y_true != 0,
                np.abs(residual) / np.abs(y_true) * 100,
                np.nan
            )
        ),
        "Bias": np.mean(residual),
        "Max_abs_error": np.max(np.abs(residual)),
    })

In [ ]:
target_names = [
    c.replace("_pred", "")
    for c in combined_clean.columns
    if c.endswith("_pred") and c != "total_load_pred"
]

best_days_all_targets = []

for target in target_names:
    daily_error = (
        combined_clean
        .groupby("issue_time")
        .apply(calculate_day_error_for_target, target_name=target)
        .reset_index()
        .sort_values("MAE")
        .reset_index(drop=True)
    )

    best_row = daily_error.iloc[0].copy()
    best_row["target"] = target
    best_days_all_targets.append(best_row)

best_days_all_targets = pd.DataFrame(best_days_all_targets)

best_days_all_targets = best_days_all_targets[
    ["target", "issue_time", "n_steps", "MAE", "RMSE", "MAPE", "Bias", "Max_abs_error"]
]

best_days_all_targets

## Concate all load predictions

### 15 Min Resolution

In [ ]:
from pathlib import Path
import pandas as pd

def profile_prediction_to_long(path, target_name, steps=96, step_minutes=15):
    df = pd.read_csv(path)

    df["Time"] = pd.to_datetime(df["Time"], errors="coerce")
    df = df.dropna(subset=["Time"]).sort_values("Time")

    rows = []

    for _, row in df.iterrows():
        issue_time = row["Time"]

        for step in range(1, steps + 1):
            true_col = f"y_tplus_{step:03d}_true"
            pred_col = f"y_tplus_{step:03d}_pred"

            rows.append({
                "issue_time": issue_time,
                "horizon_step": step,
                "Time": issue_time + pd.Timedelta(minutes=step_minutes * step),
                f"{target_name}_true": row[true_col],
                f"{target_name}_pred": row[pred_col],
            })

    return pd.DataFrame(rows)

In [ ]:
from pathlib import Path
import pandas as pd

PRED_DIR = Path(r"C:\Data_analysis\Thesis\outputs\profile_forecasting\predictions")

prediction_files = {
    "Academy": PRED_DIR / "run_20260410_143121_3cf17790_xgboost_BU_TotActPwr_Academy_profile_predictions.csv",
    "Tech_Room": PRED_DIR / "run_20260410_161400_8ef7783c_random_forest_BU_TotActPwr_Tech_Room_profile_predictions.csv",
    "SDB_EL_Substation": PRED_DIR / "run_20260410_163727_008f1b43_ridge_BU_TotActPwr_SDB_EL_Substation_profile_predictions.csv",
    #"BESS_Panel1": PRED_DIR / "run_20260410_165432_12345678_xgboost_BU_TotActPwr_BESS_Panel1_profile_predictions.csv",
}

combined = None

for name, path in prediction_files.items():
    df_long = profile_prediction_to_long(path, name)

    if combined is None:
        combined = df_long
    else:
        combined = combined.merge(
            df_long,
            on=["issue_time", "horizon_step", "Time"],
            how="outer"
        )

combined = combined.sort_values(["issue_time", "horizon_step"]).reset_index(drop=True)

combined.head()

In [ ]:
pred_cols = [c for c in combined.columns if c.endswith("_pred")]
true_cols = [c for c in combined.columns if c.endswith("_true")]

combined["total_load_pred"] = combined[pred_cols].sum(axis=1)
combined["total_load_true"] = combined[true_cols].sum(axis=1)

combined.head()

In [ ]:
# Make sure datetime columns are correct
combined["issue_time"] = pd.to_datetime(combined["issue_time"], errors="coerce")
combined["Time"] = pd.to_datetime(combined["Time"], errors="coerce")

# Temporary issue date column
combined["issue_date"] = combined["issue_time"].dt.date

# Columns that must be available for EMS validation
required_cols = [
    c for c in combined.columns
    if c.endswith("_true") or c.endswith("_pred")
]

# Find forecast issue days where at least one target is missing
bad_issue_dates = (
    combined
    .groupby("issue_date")[required_cols]
    .apply(lambda x: x.isna().any().any())
)

bad_issue_dates = bad_issue_dates[bad_issue_dates].index.tolist()

print("Bad issue dates to drop:")
print(bad_issue_dates)

# Create clean dataframe
combined_clean = combined[~combined["issue_date"].isin(bad_issue_dates)].copy()

# Remove temporary helper column
combined_clean = combined_clean.drop(columns=["issue_date"])

# Recreate total actual and predicted load
pred_cols = [c for c in combined_clean.columns if c.endswith("_pred")]
true_cols = [c for c in combined_clean.columns if c.endswith("_true")]

combined_clean["total_load_pred"] = combined_clean[pred_cols].sum(axis=1)
combined_clean["total_load_true"] = combined_clean[true_cols].sum(axis=1)

print("Before:", combined.shape)
print("After:", combined_clean.shape)
print("Dropped rows:", combined.shape[0] - combined_clean.shape[0])

combined_clean.head()

In [ ]:
import matplotlib.pyplot as plt

# Make sure Time is datetime
combined_clean["Time"] = pd.to_datetime(combined_clean["Time"], errors="coerce")

# Sort by predicted/forecasted time
df_plot = combined_clean.sort_values("Time").copy()

# All predicted load target columns
pred_cols = [c for c in df_plot.columns if c.endswith("_pred")]
true_cols = [c for c in df_plot.columns if c.endswith("_true")]

# Remove total from individual target list, because we plot total separately
target_pred_cols = [c for c in pred_cols if c != "total_load_pred"]

for pred_col in target_pred_cols:
    target_name = pred_col.replace("_pred", "")
    true_col = f"{target_name}_true"

    plt.figure(figsize=(16, 5))

    if true_col in df_plot.columns:
        plt.plot(df_plot["Time"], df_plot[true_col], label=f"{target_name} actual", linewidth=1.5)

    plt.plot(df_plot["Time"], df_plot[pred_col], label=f"{target_name} predicted", linewidth=1.5)

    plt.title(f"{target_name}: Actual vs Predicted Load Over Forecast Time")
    plt.xlabel("Forecasted Time")
    plt.ylabel("Load [kW]")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
plt.figure(figsize=(16, 5))

plt.plot(
    df_plot["Time"],
    df_plot["total_load_true"],
    label="Total actual load",
    linewidth=1.5
)

plt.plot(
    df_plot["Time"],
    df_plot["total_load_pred"],
    label="Total predicted load",
    linewidth=1.5
)

plt.title("Total Combined Load: Actual vs Predicted Over Forecast Time")
plt.xlabel("Forecasted Time")
plt.ylabel("Total Load [kW]")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Check available forecast issue times
available_issues = sorted(combined_clean["issue_time"].dropna().unique())

print("Number of forecast issue times:", len(available_issues))
print("First 5 issue times:")
print(available_issues[:5])

In [ ]:
selected_issue_time = available_issues[0]

one_day = combined_clean[
    combined_clean["issue_time"] == selected_issue_time
].copy()

one_day = one_day.sort_values("horizon_step")

In [ ]:
target_pred_cols = [
    c for c in one_day.columns
    if c.endswith("_pred") and c != "total_load_pred"
]

for pred_col in target_pred_cols:
    target_name = pred_col.replace("_pred", "")
    true_col = f"{target_name}_true"

    plt.figure(figsize=(14, 4))

    if true_col in one_day.columns:
        plt.plot(one_day["Time"], one_day[true_col], marker="o", label=f"{target_name} actual")

    plt.plot(one_day["Time"], one_day[pred_col], marker="o", label=f"{target_name} predicted")

    plt.title(f"{target_name}: Day-Ahead Forecast for Issue Time {selected_issue_time}")
    plt.xlabel("Predicted Time")
    plt.ylabel("Load [kW]")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
plt.figure(figsize=(14, 4))

plt.plot(
    one_day["Time"],
    one_day["total_load_true"],
    marker="o",
    label="Total actual load"
)

plt.plot(
    one_day["Time"],
    one_day["total_load_pred"],
    marker="o",
    label="Total predicted load"
)

plt.title(f"Total Combined Load: Day-Ahead Forecast for Issue Time {selected_issue_time}")
plt.xlabel("Predicted Time")
plt.ylabel("Total Load [kW]")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Calculate residuals for each target and total load


In [ ]:
import numpy as np
import pandas as pd

df_eval = combined_clean.copy()

df_eval["issue_time"] = pd.to_datetime(df_eval["issue_time"], errors="coerce")
df_eval["Time"] = pd.to_datetime(df_eval["Time"], errors="coerce")

# Find prediction columns
pred_cols = [c for c in df_eval.columns if c.endswith("_pred")]

# Create residual and absolute error columns
for pred_col in pred_cols:
    base_name = pred_col.replace("_pred", "")
    true_col = f"{base_name}_true"
    
    if true_col in df_eval.columns:
        df_eval[f"{base_name}_residual"] = df_eval[true_col] - df_eval[pred_col]
        df_eval[f"{base_name}_abs_error"] = df_eval[f"{base_name}_residual"].abs()
        df_eval[f"{base_name}_squared_error"] = df_eval[f"{base_name}_residual"] ** 2
        
        df_eval[f"{base_name}_ape"] = np.where(
            df_eval[true_col] != 0,
            df_eval[f"{base_name}_abs_error"] / df_eval[true_col].abs() * 100,
            np.nan
        )

df_eval.head()

In [ ]:
def calculate_day_error(group, target_name="total_load"):
    true_col = f"{target_name}_true"
    pred_col = f"{target_name}_pred"
    
    y_true = group[true_col].values
    y_pred = group[pred_col].values
    
    residual = y_true - y_pred
    
    mae = np.mean(np.abs(residual))
    rmse = np.sqrt(np.mean(residual ** 2))
    
    mape = np.mean(
        np.where(
            y_true != 0,
            np.abs(residual) / np.abs(y_true) * 100,
            np.nan
        )
    )
    
    bias = np.mean(residual)
    max_abs_error = np.max(np.abs(residual))
    
    return pd.Series({
        "n_steps": len(group),
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape,
        "Bias": bias,
        "Max_abs_error": max_abs_error,
    })


daily_total_error = (
    df_eval
    .groupby("issue_time")
    .apply(calculate_day_error, target_name="total_load")
    .reset_index()
)

daily_total_error = daily_total_error.sort_values("MAE").reset_index(drop=True)

daily_total_error.head(10)

### Get the best predicted day

In [ ]:
best_day = daily_total_error.iloc[0]

best_issue_time = best_day["issue_time"]

print("Best predicted day based on total load MAE:")
print(best_day)

In [ ]:
best_day_df = df_eval[df_eval["issue_time"] == best_issue_time].copy()
best_day_df = best_day_df.sort_values("horizon_step")

best_day_df.head()

### Plot actual vs predicted for best total-load day

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 5))

plt.plot(
    best_day_df["Time"],
    best_day_df["total_load_true"],
    marker="o",
    label="Total actual load"
)

plt.plot(
    best_day_df["Time"],
    best_day_df["total_load_pred"],
    marker="o",
    label="Total predicted load"
)

plt.title(f"Best Predicted Day - Total Load\nIssue Time: {best_issue_time}")
plt.xlabel("Predicted Time")
plt.ylabel("Total Load [kW]")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Plot residuals for best day

In [ ]:
plt.figure(figsize=(14, 4))

plt.axhline(0, linestyle="--", linewidth=1)

plt.plot(
    best_day_df["Time"],
    best_day_df["total_load_residual"],
    marker="o",
    label="Residual = Actual - Predicted"
)

plt.title(f"Residuals for Best Predicted Day - Total Load\nIssue Time: {best_issue_time}")
plt.xlabel("Predicted Time")
plt.ylabel("Residual [kW]")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Find best predicted day for each individual target

In [ ]:
target_names = [
    c.replace("_pred", "")
    for c in df_eval.columns
    if c.endswith("_pred")
]

best_days_all_targets = []

for target in target_names:
    true_col = f"{target}_true"
    pred_col = f"{target}_pred"
    
    if true_col not in df_eval.columns:
        continue
    
    daily_error = (
        df_eval
        .groupby("issue_time")
        .apply(calculate_day_error, target_name=target)
        .reset_index()
        .sort_values("MAE")
        .reset_index(drop=True)
    )
    
    best_row = daily_error.iloc[0].copy()
    best_row["target"] = target
    
    best_days_all_targets.append(best_row)

best_days_all_targets = pd.DataFrame(best_days_all_targets)

best_days_all_targets = best_days_all_targets[
    ["target", "issue_time", "n_steps", "MAE", "RMSE", "MAPE", "Bias", "Max_abs_error"]
]

best_days_all_targets

### Plot all target residuals for the best total-load day

In [ ]:
# residual_cols = [
#     c for c in best_day_df.columns
#     if c.endswith("_residual") and c != "total_load_residual"
# ]

# for residual_col in residual_cols:
#     target_name = residual_col.replace("_residual", "")
    
#     plt.figure(figsize=(14, 4))
#     plt.axhline(0, linestyle="--", linewidth=1)
    
#     plt.plot(
#         best_day_df["Time"],
#         best_day_df[residual_col],
#         marker="o",
#         label=f"{target_name} residual"
#     )
    
#     plt.title(f"{target_name} Residuals on Best Total-Load Prediction Day\nIssue Time: {best_issue_time}")
#     plt.xlabel("Predicted Time")
#     plt.ylabel("Residual [kW]")
#     plt.legend()
#     plt.grid(True, alpha=0.3)
#     plt.tight_layout()
#     plt.show()

## Load EMS Scenarioas

In [ ]:
import numpy as np
import pandas as pd

# Make sure datetime columns are correct
df_step2["issue_time"] = pd.to_datetime(df_step2["issue_time"], errors="coerce")
df_step2["Time"] = pd.to_datetime(df_step2["Time"], errors="coerce")

# Make sure forecast errors exist
df_step2["forecast_residual"] = df_step2["total_load_true"] - df_step2["total_load_pred"]
df_step2["forecast_abs_error"] = df_step2["forecast_residual"].abs()
df_step2["forecast_squared_error"] = df_step2["forecast_residual"] ** 2

df_step2["forecast_ape"] = np.where(
    df_step2["total_load_true"] != 0,
    df_step2["forecast_abs_error"] / df_step2["total_load_true"].abs() * 100,
    np.nan
)

# Calculate daily forecast error over full 96-step horizon
forecast_day_error = (
    df_step2
    .groupby("issue_time")
    .apply(lambda g: pd.Series({
        "n_steps": len(g),
        "MAE": g["forecast_abs_error"].mean(),
        "RMSE": np.sqrt(g["forecast_squared_error"].mean()),
        "MAPE": g["forecast_ape"].mean(),
        "Bias": g["forecast_residual"].mean(),
        "Max_abs_error": g["forecast_abs_error"].max(),
    }))
    .reset_index()
)

# Keep only complete 96-step forecast days
forecast_day_error = forecast_day_error[forecast_day_error["n_steps"] == 96].copy()

# Best and worst day based on total-load forecast MAE
best_day_row = forecast_day_error.sort_values("MAE", ascending=True).iloc[0]
worst_day_row = forecast_day_error.sort_values("MAE", ascending=False).iloc[0]

best_issue_time = best_day_row["issue_time"]
worst_issue_time = worst_day_row["issue_time"]

print("Best forecasted day:")
print(best_day_row)

print("\nWorst forecasted day:")
print(worst_day_row)

In [ ]:
best_day = (
    df_step2[df_step2["issue_time"] == best_issue_time]
    .copy()
    .sort_values("horizon_step")
)

worst_day = (
    df_step2[df_step2["issue_time"] == worst_issue_time]
    .copy()
    .sort_values("horizon_step")
)

print("Best issue time:", best_issue_time)
print("Worst issue time:", worst_issue_time)

print("Best day shape:", best_day.shape)
print("Worst day shape:", worst_day.shape)

In [ ]:
import matplotlib.pyplot as plt

def plot_load_input_comparison(one_day, title_prefix):
    plt.figure(figsize=(14, 5))

    plt.plot(
        one_day["Time"],
        one_day["total_load_true"],
        marker="o",
        label="Actual load"
    )

    plt.plot(
        one_day["Time"],
        one_day["total_load_pred"],
        marker="o",
        label="Forecast load"
    )

    plt.plot(
        one_day["Time"],
        one_day["constant_fixed"],
        linestyle="--",
        label="Option A: Fixed constant"
    )

    plt.plot(
        one_day["Time"],
        one_day["constant_hold"],
        linestyle=":",
        linewidth=3,
        label="Option B: Constant-hold"
    )

    issue_time = one_day["issue_time"].iloc[0]

    plt.title(f"{title_prefix}\nIssue Time: {issue_time}")
    plt.xlabel("Predicted Time")
    plt.ylabel("Total Load [kW]")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_load_input_comparison(best_day, "Best Forecasted Day: Actual vs Forecast vs Constant EMS Inputs")
plot_load_input_comparison(worst_day, "Worst Forecasted Day: Actual vs Forecast vs Constant EMS Inputs")

In [ ]:
def plot_residual_comparison(one_day, title_prefix):
    plt.figure(figsize=(14, 5))

    plt.axhline(0, linestyle="--", linewidth=1)

    plt.plot(
        one_day["Time"],
        one_day["forecast_residual"],
        marker="o",
        label="Forecast residual"
    )

    plt.plot(
        one_day["Time"],
        one_day["fixed_residual"],
        marker="o",
        label="Option A fixed residual"
    )

    plt.plot(
        one_day["Time"],
        one_day["hold_residual"],
        marker="o",
        label="Option B hold residual"
    )

    issue_time = one_day["issue_time"].iloc[0]

    plt.title(f"{title_prefix}\nIssue Time: {issue_time}")
    plt.xlabel("Predicted Time")
    plt.ylabel("Residual [kW] = Actual - Input")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_residual_comparison(best_day, "Residual Comparison for Best Forecasted Day")
plot_residual_comparison(worst_day, "Residual Comparison for Worst Forecasted Day")

In [ ]:
def plot_absolute_error_comparison(one_day, title_prefix):
    plt.figure(figsize=(14, 5))

    plt.plot(
        one_day["Time"],
        one_day["forecast_abs_error"],
        marker="o",
        label="Forecast absolute error"
    )

    plt.plot(
        one_day["Time"],
        one_day["fixed_abs_error"],
        marker="o",
        label="Option A fixed absolute error"
    )

    plt.plot(
        one_day["Time"],
        one_day["hold_abs_error"],
        marker="o",
        label="Option B hold absolute error"
    )

    issue_time = one_day["issue_time"].iloc[0]

    plt.title(f"{title_prefix}\nIssue Time: {issue_time}")
    plt.xlabel("Predicted Time")
    plt.ylabel("Absolute Error [kW]")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df_step2 = combined_clean.copy()

df_step2["issue_time"] = pd.to_datetime(df_step2["issue_time"], errors="coerce")
df_step2["Time"] = pd.to_datetime(df_step2["Time"], errors="coerce")

df_step2 = df_step2.sort_values(["issue_time", "horizon_step"]).reset_index(drop=True)

In [ ]:
plot_absolute_error_comparison(best_day, "Absolute Error Comparison for Best Forecasted Day")
plot_absolute_error_comparison(worst_day, "Absolute Error Comparison for Worst Forecasted Day")

In [ ]:
def calculate_energy_error_summary(one_day, step_minutes=15):
    step_hours = step_minutes / 60

    d = one_day.copy()

    d["actual_energy_kWh"] = d["total_load_true"] * step_hours
    d["forecast_energy_kWh"] = d["total_load_pred"] * step_hours
    d["fixed_energy_kWh"] = d["constant_fixed"] * step_hours
    d["hold_energy_kWh"] = d["constant_hold"] * step_hours

    d["forecast_energy_error_kWh"] = d["actual_energy_kWh"] - d["forecast_energy_kWh"]
    d["fixed_energy_error_kWh"] = d["actual_energy_kWh"] - d["fixed_energy_kWh"]
    d["hold_energy_error_kWh"] = d["actual_energy_kWh"] - d["hold_energy_kWh"]

    actual_energy = d["actual_energy_kWh"].sum()

    summary = pd.DataFrame({
        "case": [
            "Forecast load",
            "Option A: fixed constant",
            "Option B: constant-hold",
        ],
        "actual_energy_kWh": [
            actual_energy,
            actual_energy,
            actual_energy,
        ],
        "input_energy_kWh": [
            d["forecast_energy_kWh"].sum(),
            d["fixed_energy_kWh"].sum(),
            d["hold_energy_kWh"].sum(),
        ],
        "signed_energy_error_kWh": [
            d["forecast_energy_error_kWh"].sum(),
            d["fixed_energy_error_kWh"].sum(),
            d["hold_energy_error_kWh"].sum(),
        ],
        "absolute_energy_error_kWh": [
            d["forecast_energy_error_kWh"].abs().sum(),
            d["fixed_energy_error_kWh"].abs().sum(),
            d["hold_energy_error_kWh"].abs().sum(),
        ],
    })

    summary["signed_energy_error_%"] = (
        summary["signed_energy_error_kWh"] / summary["actual_energy_kWh"] * 100
    )

    summary["absolute_energy_error_%"] = (
        summary["absolute_energy_error_kWh"] / summary["actual_energy_kWh"] * 100
    )

    return summary, d

In [ ]:
best_energy_summary, best_day_energy = calculate_energy_error_summary(best_day)
worst_energy_summary, worst_day_energy = calculate_energy_error_summary(worst_day)

print("Best forecasted day energy error:")
display(best_energy_summary)

print("Worst forecasted day energy error:")
display(worst_energy_summary)


In [ ]:
def plot_cumulative_energy_error(day_energy_df, title_prefix):
    d = day_energy_df.copy()

    d["forecast_cum_energy_error_kWh"] = d["forecast_energy_error_kWh"].cumsum()
    d["fixed_cum_energy_error_kWh"] = d["fixed_energy_error_kWh"].cumsum()
    d["hold_cum_energy_error_kWh"] = d["hold_energy_error_kWh"].cumsum()

    plt.figure(figsize=(14, 5))

    plt.axhline(0, linestyle="--", linewidth=1)

    plt.plot(
        d["Time"],
        d["forecast_cum_energy_error_kWh"],
        marker="o",
        label="Forecast cumulative energy error"
    )

    plt.plot(
        d["Time"],
        d["fixed_cum_energy_error_kWh"],
        marker="o",
        label="Option A fixed cumulative energy error"
    )

    plt.plot(
        d["Time"],
        d["hold_cum_energy_error_kWh"],
        marker="o",
        label="Option B hold cumulative energy error"
    )

    issue_time = d["issue_time"].iloc[0]

    plt.title(f"{title_prefix}\nIssue Time: {issue_time}")
    plt.xlabel("Predicted Time")
    plt.ylabel("Cumulative Energy Error [kWh]")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_cumulative_energy_error(best_day_energy, "Cumulative Energy Error for Best Forecasted Day")
plot_cumulative_energy_error(worst_day_energy, "Cumulative Energy Error for Worst Forecasted Day")

In [ ]:
best_energy_summary["forecast_quality_day"] = "Best forecasted day"
worst_energy_summary["forecast_quality_day"] = "Worst forecasted day"

energy_summary_best_worst = pd.concat(
    [best_energy_summary, worst_energy_summary],
    ignore_index=True
)

energy_summary_best_worst = energy_summary_best_worst[
    [
        "forecast_quality_day",
        "case",
        "actual_energy_kWh",
        "input_energy_kWh",
        "signed_energy_error_kWh",
        "absolute_energy_error_kWh",
        "signed_energy_error_%",
        "absolute_energy_error_%",
    ]
]

energy_summary_best_worst

In [ ]:
global_error_comparison = pd.DataFrame({
    "case": [
        "Forecast load",
        "Option A: fixed constant",
        "Option B: constant-hold",
    ],
    "MAE": [
        df_step2["forecast_abs_error"].mean(),
        df_step2["fixed_abs_error"].mean(),
        df_step2["hold_abs_error"].mean(),
    ],
    "RMSE": [
        np.sqrt((df_step2["forecast_residual"] ** 2).mean()),
        np.sqrt((df_step2["fixed_residual"] ** 2).mean()),
        np.sqrt((df_step2["hold_residual"] ** 2).mean()),
    ],
    "Bias": [
        df_step2["forecast_residual"].mean(),
        df_step2["fixed_residual"].mean(),
        df_step2["hold_residual"].mean(),
    ],
    "Max_abs_error": [
        df_step2["forecast_abs_error"].max(),
        df_step2["fixed_abs_error"].max(),
        df_step2["hold_abs_error"].max(),
    ],
})

global_error_comparison